In [ ]:
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification, CLIPProcessor, CLIPModel
from pathlib import Path
from tqdm import tqdm
import os

# --- 1. 설정 (Configuration) ---
print("1. 설정을 초기화합니다...")

# [사용자 설정] 모든 파일 경로를 여기에 모아서 관리합니다.
DEV_TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_for_clip.csv"
SAMPLE_SUBMISSION_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv"
BASE_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"
# 최종적으로 저장될 제출 파일 이름
FINAL_SUBMISSION_PATH = "clip_submission_final.csv"

# 디바이스 설정 (GPU가 있으면 자동으로 사용)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {DEVICE}")

# --- 2. 모델 및 프로세서 로드 ---
print("\n2. CLIP 모델과 프로세서를 로드합니다...")
# 제로샷 이미지 분류에 특화된 CLIP 모델을 로드합니다.
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
# model = AutoModelForZeroShotImageClassification.from_pretrained(MODEL_NAME).to(DEVICE)
# 모델을 추론 모드로 설정합니다.
model.eval()
print("모델 로드 완료.")

# --- 3. 데이터 로드 및 추론 루프 ---
print("\n3. 데이터 로드를 시작합니다...")
# 추론할 이미지와 질문 정보가 담긴 dev_test.csv 로드
try:
    df_test = pd.read_csv(DEV_TEST_CSV_PATH)
except FileNotFoundError:
    print(f"오류: {DEV_TEST_CSV_PATH} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 결과를 채워넣을 sample_submission.csv 로드
try:
    df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
except FileNotFoundError:
    print(f"오류: {SAMPLE_SUBMISSION_PATH} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 예측된 인덱스(0,1,2,3)를 정답 레이블(A,B,C,D)로 변환하기 위한 딕셔너리
index_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

print(f"\n총 {len(df_test)}개의 데이터에 대한 추론을 시작합니다...")
# tqdm을 사용하여 진행 상황을 시각적으로 표시
for index, row in tqdm(df_test.iterrows(), total=df_test.shape[0], desc="추론 진행률"):
    
    test_id = row['ID']
    relative_img_path = row['img_path']
    choices = [row['A'], row['B'], row['C'], row['D']]

    # 전체 이미지 경로 구성
    image_filename = os.path.basename(relative_img_path)
    full_image_path = Path(BASE_IMAGE_DIR) / image_filename

    try:
        # 이미지 열기
        image = Image.open(full_image_path).convert("RGB")
        
        # CLIP 모델 입력 준비: 이미지와 4개의 선택지 텍스트를 함께 전달
        inputs = processor(images=image, text=choices, return_tensors="pt", padding=True)
        # 모든 입력을 현재 디바이스(GPU 또는 CPU)로 이동
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        # 그래디언트 계산 비활성화 (추론 시 필수)
        with torch.no_grad():
            outputs = model(**inputs)
        
        # 이미지와 각 선택지(텍스트) 간의 유사도 점수
        logits_per_image = outputs.logits_per_image.cpu() # 결과를 CPU로 이동

        # 가장 높은 점수를 가진 선택지의 인덱스 찾기
        predicted_index = logits_per_image.argmax(-1).item()
        
        # 인덱스를 'A', 'B', 'C', 'D' 문자로 변환
        prediction_label = index_to_label[predicted_index]

        # df_submission에서 현재 ID와 일치하는 행을 찾아 'answer' 컬럼을 업데이트
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = prediction_label

    except FileNotFoundError:
        print(f"\n경고: ID '{test_id}'의 이미지 파일을 찾을 수 없습니다. 경로: {full_image_path}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'
    except Exception as e:
        print(f"\n오류: ID '{test_id}' 처리 중 예외 발생: {e}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'

# --- 4. 결과 저장 및 출력 ---
# 예측값으로 모두 채워진 df_submission 데이터프레임을 최종 파일로 저장
df_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

print("\n" + "="*50)
print("추론이 완료되었습니다.")
print(f"결과가 {FINAL_SUBMISSION_PATH} 파일에 저장되었습니다.")
print("\n최종 제출 파일 미리보기 (상위 5개):")
print(df_submission.head())
print("="*50)

1. 설정을 초기화합니다...
사용 디바이스: cuda

2. CLIP 모델과 프로세서를 로드합니다...
모델 로드 완료.

3. 데이터 로드를 시작합니다...

총 60개의 데이터에 대한 추론을 시작합니다...


추론 진행률:  17%|█▋        | 10/60 [00:00<00:01, 30.97it/s]


오류: ID 'TEST_000' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_001' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_002' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_003' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_004' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_005' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_006' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_007' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_008' 처리 중 

추론 진행률:  42%|████▏     | 25/60 [00:00<00:00, 53.67it/s]


오류: ID 'TEST_014' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_015' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_016' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_017' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_018' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_019' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_020' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_021' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_022' 처리 중 

추론 진행률:  67%|██████▋   | 40/60 [00:00<00:00, 62.40it/s]


오류: ID 'TEST_029' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_030' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_031' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_032' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_033' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_034' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_035' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_036' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_037' 처리 중 

추론 진행률: 100%|██████████| 60/60 [00:01<00:00, 54.40it/s]


오류: ID 'TEST_045' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_046' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_047' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_048' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_049' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_050' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_051' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_052' 처리 중 예외 발생: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

오류: ID 'TEST_053' 처리 중 

In [4]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 151,277,313개 (151.28M)
학습 가능한 파라미터: 151,277,313개 (151.28M)
